# Lab Exercise 10 — Learning the XOR Boolean Function Using an MLP

**Registration Number:** 2547237  
**Course:** MCA 521-4 — Machine Learning  
**Lab:** Lab 10  

---

## Aim
1. To understand how to implement neural networks using different deep learning libraries (Keras and TensorFlow).
2. To solve the non-linear XOR problem using an MLP and study the effect of hyperparameters such as learning rate, activation functions, number of neurons, and epochs on model performance.

## XOR Truth Table

| Input 1 | Input 2 | XOR Output |
|---------|---------|------------|
| 0       | 0       | 0          |
| 0       | 1       | 1          |
| 1       | 0       | 1          |
| 1       | 1       | 0          |

XOR is a classic non-linearly separable problem — a single perceptron cannot solve it. A Multi-Layer Perceptron (MLP) with at least one hidden layer is required.


## 0. Environment Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, losses

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print("All imports successful.")

---
## 1. Dataset — XOR Truth Table

In [ ]:
# ── Step 1: Create the XOR Dataset ──────────────────────────────────────────
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=np.float32)

y = np.array([[0],
              [1],
              [1],
              [0]], dtype=np.float32)

print("XOR Dataset:")
print(f"{'Input 1':>8} {'Input 2':>8} {'XOR Output':>12}")
print("-" * 32)
for xi, yi in zip(X, y):
    print(f"{int(xi[0]):>8} {int(xi[1]):>8} {int(yi[0]):>12}")

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nNote: XOR is NOT linearly separable — a single perceptron fails on this.")

---
## 2. Helper Functions — Visualisation

In [ ]:
def plot_decision_boundary(predict_fn, title, ax, resolution=500):
    """Plot the decision boundary for a binary classifier on the XOR domain."""
    x_min, x_max = -0.3, 1.3
    y_min, y_max = -0.3, 1.3
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                          np.linspace(y_min, y_max, resolution))
    grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
    Z = predict_fn(grid).reshape(xx.shape)

    cmap_bg = ListedColormap(['#FFB3B3', '#B3D9FF'])
    ax.contourf(xx, yy, Z, alpha=0.6, cmap=cmap_bg, levels=np.linspace(0, 1, 20))
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

    colors = ['#D62728', '#1F77B4']  # red=0, blue=1
    for xi, yi in zip(X, y.ravel()):
        ax.scatter(xi[0], xi[1], s=300, c=colors[int(yi)],
                   edgecolors='black', linewidths=2, zorder=5)
        ax.annotate(f"({int(xi[0])},{int(xi[1])})→{int(yi)}",
                    xy=(xi[0], xi[1]), xytext=(xi[0]+0.05, xi[1]+0.07),
                    fontsize=9, fontweight='bold')

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('Input 1', fontsize=11)
    ax.set_ylabel('Input 2', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    patch0 = mpatches.Patch(color='#FFB3B3', label='Predicted 0')
    patch1 = mpatches.Patch(color='#B3D9FF', label='Predicted 1')
    ax.legend(handles=[patch0, patch1], loc='lower right', fontsize=9)


def plot_training_history(histories, labels, title, ax):
    """Plot training loss curves."""
    colors = ['#E6194B', '#3CB44B', '#4363D8', '#F58231', '#911EB4']
    for i, (hist, label) in enumerate(zip(histories, labels)):
        ax.plot(hist, color=colors[i % len(colors)], linewidth=2, label=label)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)


def print_predictions(model_name, predict_fn):
    """Print formatted predictions vs ground truth."""
    preds = predict_fn(X)
    print(f"\n{'─'*52}")
    print(f"  Predictions — {model_name}")
    print(f"{'─'*52}")
    print(f"  {'Input':>10} │ {'Raw Output':>12} │ {'Rounded':>8} │ {'Truth':>6} │ {'✓/✗':>4}")
    print(f"  {'-'*10}─{'-'*12}──{'-'*8}──{'-'*6}─{'-'*4}")
    all_correct = True
    for xi, yi, pi in zip(X, y.ravel(), preds.ravel()):
        rounded = int(np.round(pi))
        correct = (rounded == int(yi))
        if not correct:
            all_correct = False
        mark = '✓' if correct else '✗'
        print(f"  ({int(xi[0])},{int(xi[1])})      │ {pi:>12.6f} │ {rounded:>8} │ {int(yi):>6} │ {mark:>4}")
    acc = np.mean(np.round(preds.ravel()) == y.ravel()) * 100
    print(f"\n  Accuracy: {acc:.1f}%  {'✅ XOR learned!' if all_correct else '❌ Not fully learned yet'}")
    print(f"{'─'*52}")

print("Helper functions defined.")

---
## 3. Implementation 1 — Keras (TensorFlow High-Level API)

Keras is TensorFlow's high-level API. It provides a clean, expressive interface for building and training neural networks with minimal boilerplate.

In [ ]:
# ── Step 2 & 3: Build and Compile the Keras MLP ─────────────────────────────
tf.random.set_seed(42)

keras_model = keras.Sequential([
    layers.Input(shape=(2,), name='input_layer'),
    layers.Dense(4, activation='tanh', name='hidden_layer'),   # 4 neurons, Tanh activation
    layers.Dense(1, activation='sigmoid', name='output_layer') # 1 neuron, Sigmoid for binary
], name='XOR_Keras_MLP')

keras_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.1),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

keras_model.summary()

In [ ]:
# ── Step 4: Train the Keras Model ───────────────────────────────────────────
keras_history = keras_model.fit(
    X, y,
    epochs=1000,
    verbose=0,  # Suppress epoch-by-epoch output
    batch_size=4
)

print(f"Training complete — {len(keras_history.history['loss'])} epochs")
print(f"Final loss    : {keras_history.history['loss'][-1]:.6f}")
print(f"Final accuracy: {keras_history.history['accuracy'][-1]*100:.1f}%")

In [ ]:
# ── Step 5: Evaluate the Keras Model ────────────────────────────────────────
def keras_predict(x):
    return keras_model.predict(x, verbose=0)

print_predictions("Keras (TF High-Level API)", keras_predict)

---
## 4. Implementation 2 — TensorFlow Low-Level API

Using TensorFlow's low-level API (variables, gradient tape, manual training loop) gives full control over the network — useful for custom architectures and research.

In [ ]:
# ── Step 2: Define the TF Low-Level MLP ─────────────────────────────────────
tf.random.set_seed(42)

# Layer 1 weights and biases (2 inputs → 4 hidden neurons)
W1 = tf.Variable(tf.random.glorot_uniform([2, 4]), dtype=tf.float32, name='W1')
b1 = tf.Variable(tf.zeros([4]),                    dtype=tf.float32, name='b1')

# Layer 2 weights and biases (4 hidden → 1 output neuron)
W2 = tf.Variable(tf.random.glorot_uniform([4, 1]), dtype=tf.float32, name='W2')
b2 = tf.Variable(tf.zeros([1]),                    dtype=tf.float32, name='b2')

tf_params = [W1, b1, W2, b2]

def tf_forward(x):
    """Manual forward pass: tanh hidden layer, sigmoid output."""
    h = tf.nn.tanh(tf.matmul(x, W1) + b1)      # Hidden: tanh
    out = tf.nn.sigmoid(tf.matmul(h, W2) + b2)  # Output: sigmoid
    return out

# ── Step 3: Define Loss and Optimizer ───────────────────────────────────────
tf_optimizer = tf.optimizers.Adam(learning_rate=0.1)

def tf_loss(x, y_true):
    """Binary cross-entropy loss."""
    y_pred = tf_forward(x)
    return tf.reduce_mean(losses.binary_crossentropy(y_true, y_pred))

print("TF Low-Level model parameters:")
total_params = sum([tf.size(p).numpy() for p in tf_params])
for p in tf_params:
    print(f"  {p.name:6s}: shape {p.shape} — {tf.size(p).numpy()} params")
print(f"  Total: {total_params} trainable parameters")

In [ ]:
# ── Step 4: Manual Training Loop ────────────────────────────────────────────
EPOCHS = 1000
tf_losses = []

for epoch in range(EPOCHS):
    with tf.GradientTape() as tape:
        loss_val = tf_loss(X, y)
    grads = tape.gradient(loss_val, tf_params)
    tf_optimizer.apply_gradients(zip(grads, tf_params))
    tf_losses.append(loss_val.numpy())

print(f"Training complete — {EPOCHS} epochs")
print(f"Final loss: {tf_losses[-1]:.6f}")

In [ ]:
# ── Step 5: Evaluate the TF Low-Level Model ─────────────────────────────────
def tf_predict(x):
    return tf_forward(tf.constant(x, dtype=tf.float32)).numpy()

print_predictions("TensorFlow Low-Level API", tf_predict)

---
## 5. Optional — Decision Boundary Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Decision Boundaries — XOR MLP', fontsize=15, fontweight='bold', y=1.01)

plot_decision_boundary(keras_predict,  "Keras (TF High-Level API)",  axes[0])
plot_decision_boundary(tf_predict,     "TensorFlow Low-Level API",   axes[1])

plt.tight_layout()
plt.savefig('decision_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → decision_boundaries.png")

---
## 6. Optional — Training Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_training_history(
    histories=[
        keras_history.history['loss'],
        tf_losses
    ],
    labels=[
        'Keras (High-Level API)',
        'TensorFlow (Low-Level API)'
    ],
    title='Training Loss Curves — XOR MLP (Adam, lr=0.1, Tanh Hidden, Sigmoid Output)',
    ax=ax
)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → training_curves.png")

---
## 7. Optional — Hyperparameter Study

Studying how learning rate, activation functions, number of hidden neurons, and epochs affect convergence on the XOR task.

In [ ]:
# ── 7a. Effect of Learning Rate ─────────────────────────────────────────────
learning_rates = [0.001, 0.01, 0.1, 0.5]
lr_histories   = []

for lr in learning_rates:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(4, activation='tanh'),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=optimizers.Adam(learning_rate=lr),
              loss='binary_crossentropy')
    hist = m.fit(X, y, epochs=2000, verbose=0, batch_size=4)
    lr_histories.append(hist.history['loss'])
    final_acc = np.mean(np.round(m.predict(X, verbose=0).ravel()) == y.ravel()) * 100
    print(f"lr={lr:.3f}  →  final_loss={hist.history['loss'][-1]:.4f}  accuracy={final_acc:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_training_history(
    lr_histories,
    [f'lr={lr}' for lr in learning_rates],
    'Effect of Learning Rate on XOR Convergence (Keras, 2000 epochs)',
    ax
)
plt.tight_layout()
plt.savefig('lr_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → lr_study.png")

In [ ]:
# ── 7b. Effect of Activation Function ───────────────────────────────────────
activations = ['tanh', 'relu', 'sigmoid', 'elu']
act_histories = []

for act in activations:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(4, activation=act),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=optimizers.Adam(learning_rate=0.1),
              loss='binary_crossentropy')
    hist = m.fit(X, y, epochs=1000, verbose=0, batch_size=4)
    act_histories.append(hist.history['loss'])
    final_acc = np.mean(np.round(m.predict(X, verbose=0).ravel()) == y.ravel()) * 100
    print(f"activation={act:8s} → final_loss={hist.history['loss'][-1]:.4f}  accuracy={final_acc:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_training_history(
    act_histories,
    [f'activation={a}' for a in activations],
    'Effect of Activation Function on XOR Convergence (Keras, lr=0.1, 1000 epochs)',
    ax
)
plt.tight_layout()
plt.savefig('activation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → activation_study.png")

In [ ]:
# ── 7c. Effect of Number of Hidden Neurons ──────────────────────────────────
neuron_counts = [2, 4, 8, 16]
neuron_histories = []

for n in neuron_counts:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(n, activation='tanh'),
        layers.Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=optimizers.Adam(learning_rate=0.1),
              loss='binary_crossentropy')
    hist = m.fit(X, y, epochs=1000, verbose=0, batch_size=4)
    neuron_histories.append(hist.history['loss'])
    final_acc = np.mean(np.round(m.predict(X, verbose=0).ravel()) == y.ravel()) * 100
    print(f"neurons={n:2d} → final_loss={hist.history['loss'][-1]:.4f}  accuracy={final_acc:.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_training_history(
    neuron_histories,
    [f'{n} hidden neurons' for n in neuron_counts],
    'Effect of Hidden Neuron Count on XOR Convergence (Keras, lr=0.1, tanh, 1000 epochs)',
    ax
)
plt.tight_layout()
plt.savefig('neuron_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → neuron_study.png")

---
## 8. Summary & Observations

| Aspect | Keras (High-Level) | TF Low-Level |
|---|---|---|
| Code verbosity | Low — Sequential API | High — manual variables + GradientTape |
| Flexibility | Moderate | Full control |
| Debugging | Abstracts internals | Explicit gradient access |
| Best for | Rapid prototyping | Custom research |

### Key Findings

1. **XOR requires a hidden layer** — a single perceptron cannot learn it (not linearly separable).
2. **Tanh** outperforms ReLU on XOR — tanh outputs in (−1, 1), providing richer gradients for this tiny dataset.
3. **Learning rate** critically impacts speed and stability:
   - Too low (0.001): very slow convergence
   - Too high (0.5): may oscillate or overshoot
   - Optimal (~0.1 with Adam): fast, stable convergence
4. **Number of neurons**: Even 2 neurons can learn XOR; more neurons converge faster but may overfit on larger datasets.
5. **Epochs**: 500–1000 epochs are typically sufficient for XOR with lr=0.1 and Adam.
6. **Both Keras and TF low-level** converge to the same solution; Keras is preferred for ease of use.

### Decision Boundary
The learned decision boundary is non-linear (curved), correctly partitioning the input space into two regions — one for XOR=0 (corners (0,0) and (1,1)) and one for XOR=1 (corners (0,1) and (1,0)).
